# 34. Saving and loading complete pipelines
A model file must preserve the preprocessing needed for predictions. Saving only the classifier loses learned scaling, encodings, and column handling. Pickle-based formats such as joblib can execute code when loaded, so load only artifacts from sources you trust. Compatibility across arbitrary library versions is not guaranteed.

Fit the selected complete pipeline, record dependency versions, and serialize it. Reload it and compare predictions on a fixed input. Keep the input schema and model metadata alongside it. This chapter checks artifact round-tripping; Chapter 35 checks input validation and a clean reproducible workflow.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
import joblib
import numpy as np
model.fit(X_dev, y_dev) # Fit a complete pipeline.
before = model.predict_proba(X_test[:5]) # Capture a fixed prediction reference.
joblib.dump(model, "course_pipeline.joblib") # Save only to a file you control.
loaded = joblib.load("course_pipeline.joblib") # Load this known, self-created artifact.
after = loaded.predict_proba(X_test[:5]) # Repeat the same input.
print("Predictions match:", np.allclose(before, after)) # Check numeric consistency.
assert np.allclose(before, after) # Fail if the round trip changes predictions.

## Feature names and version compatibility are part of the artifact
A fitted estimator can record feature_names_in_ when trained on string-named DataFrame columns. Predicting with an unnamed array may emit a warning; supplying named columns in the wrong order can raise an error. Do not silence those signals without checking semantics. Use public estimator behavior rather than relying on a private function named __sklearn_check_feature_names. Record versions and compare known test-record predictions after loading; pickle/joblib formats do not promise arbitrary cross-version compatibility. If source or dependencies changed, retraining in the target environment may be safer than forcing a load.

In [ ]:
import warnings
import pandas as pd
from sklearn.linear_model import LinearRegression
X = pd.DataFrame({"hours": [1., 2., 3., 4.], "distance": [3., 5., 4., 8.]})
y = [4., 7., 8., 12.]
model = LinearRegression().fit(X, y) # Fit with explicit feature names.
print("Expected names:", model.feature_names_in_.tolist())
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model.predict(X.to_numpy()) # Unnamed input loses an important contract check.
    print("Unnamed-input warnings:", len(caught))
try:
    model.predict(X[["distance", "hours"]]) # Named columns appear in a different order.
except ValueError:
    print("Reordered named columns: rejected")
assert model.feature_names_in_.tolist() == ["hours", "distance"]

### Reading the result
The warning is useful evidence of a missing input contract. An array with reversed columns could still produce a number without telling you it is semantically wrong. Preserve names and validate units, not just the number of columns. Never load an untrusted pickle-based artifact to inspect what is inside it.

### Transfer exercise
Add a model manifest containing feature names, input units, package versions, training-data reference, and fixed record/prediction checks.

## Your turn
Save a mixed-data pipeline from an earlier project and record Python and package versions. Reload it and compare predictions on a fixed raw record.

Try the task before opening the solution on the course website.